<a href="https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/MNPS_Job_Classification_Likelihood_Scorer_v3_1_IMPROVED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNPS Job Classification Likelihood Scorer v3.1 - IMPROVED

This notebook calculates likelihood scores for job classifications, incorporating confidence, borderline detection, and accurate salary mapping.

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import warnings
import datetime
import json
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

# Download required NLTK data (uncomment if running for the first time)
# nltk.download('punkt')
# nltk.download('stopwords')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ Dependencies and libraries loaded successfully!")

## 2. Configuration Parameters

In [ ]:
# Human Baseline Accuracy Range
HUMAN_BASELINE_MIN = 88
HUMAN_BASELINE_MAX = 94
HUMAN_BASELINE_TYPICAL = (HUMAN_BASELINE_MIN + HUMAN_BASELINE_MAX) / 2

# Component Weights (must sum to 1.0)
WEIGHT_KSAC_SIMILARITY = 0.40
WEIGHT_SALARY_IMPACT = 0.20
WEIGHT_TIME_IMPACT = 0.10
WEIGHT_SUBGROUP = 0.15
WEIGHT_JUSTIFICATION = 0.15

# Enhanced Scoring Configuration
CONFIDENCE_THRESHOLD = 0.6
ENABLE_BORDERLINE_DETECTION = True
ENABLE_JD_SUGGESTIONS = True

# Review Priority Categories (Aligned with Report Logic)
REVIEW_PRIORITY_CRITICAL = "Critical" # Below Human (likelihood < 4.0) and NOT borderline (or no suggestion needed)
REVIEW_PRIORITY_HIGH = "High" # Below Human (likelihood < 4.0) AND borderline (or low confidence with suggestion needed)
REVIEW_PRIORITY_MEDIUM = "Medium" # Human-level (4.0 <= likelihood < 4.5) AND low confidence
REVIEW_PRIORITY_LOW = "Low" # Human-level (4.0 <= likelihood < 4.5) AND high confidence, OR Above Human (likelihood >= 4.5)

print("✅ Configuration parameters set.")
print(f"Human Baseline: {HUMAN_BASELINE_MIN}%-{HUMAN_BASELINE_MAX}%")
print(f"Confidence Threshold: {CONFIDENCE_THRESHOLD}")
print(f"Review Priorities: Critical, High, Medium, Low")

## 3. Load and Prepare Resource Data

In [ ]:
# Role Groups Mapping (Simplified based on provided file structure)
role_groups_data = {
    "Group Name": [
        "Management & Supervision",
        "Program Coordination",
        "Data Analysis & Research",
        "Specialized Content Experts",
        "Technical Support & Maintenance",
        "Direct Instructional Staff",
        "Instructional Support & Development",
        "Specialized Student Services",
        "Program Communication and Community Relations",
        "Financial Operations, Planning and Procurement"
    ],
    "Roles in Group": [
        "Manager,Supervisor,Director",
        "Coordinator,Liaison",
        "Analyst,Researcher",
        "Specialist,Expert",
        "Technician,Technician Support",
        "Instructor,Teacher",
        "Coach,Instructional Coach",
        "Counselor,Social Worker",
        "Community Liaison,Outreach Coordinator",
        "Accountant,Financial Analyst"
    ]
}
role_groups_df = pd.DataFrame(role_groups_data)

# Salary Data Mapping (Simplified based on provided file structure and context)
salary_data_raw = [
    {"Major Role Grouping": "Management & Supervision", "Min Annual Salary": 70000, "Average Annual Salary": 98780.31, "Max Annual Salary": 130000},
    {"Major Role Grouping": "Program Coordination", "Min Annual Salary": 40000, "Average Annual Salary": 57402.72, "Max Annual Salary": 75000}, # Average of Executive and Mid-level Coordinators
    {"Major Role Grouping": "Data Analysis & Research", "Min Annual Salary": 50000, "Average Annual Salary": 85170.35, "Max Annual Salary": 100000}, # Management level for Analysts
    {"Major Role Grouping": "Specialized Content Experts", "Min Annual Salary": 55000, "Average Annual Salary": 77325.89, "Max Annual Salary": 90000}, # Senior level for Specialists
    {"Major Role Grouping": "Technical Support & Maintenance", "Min Annual Salary": 45000, "Average Annual Salary": 61795.46, "Max Annual Salary": 70000}, # Senior level for Technicians
    {"Major Role Grouping": "Direct Instructional Staff", "Min Annual Salary": 40000, "Average Annual Salary": 78890.12, "Max Annual Salary": 85000}, # Senior level for Instructors
    {"Major Role Grouping": "Instructional Support & Development", "Min Annual Salary": 35000, "Average Annual Salary": 42285.11, "Max Annual Salary": 50000}, # Mid level for Coaches
    {"Major Role Grouping": "Specialized Student Services", "Min Annual Salary": 45000, "Average Annual Salary": 61447.30, "Max Annual Salary": 75000}, # Average for Counselors/Social Workers
    {"Major Role Grouping": "Program Communication and Community Relations", "Min Annual Salary": 50000, "Average Annual Salary": 73351.20, "Max Annual Salary": 85000}, # Senior level for Liaisons
    {"Major Role Grouping": "Financial Operations, Planning and Procurement", "Min Annual Salary": 55000, "Average Annual Salary": 85462.72, "Max Annual Salary": 100000} # Average for Analysts/Accts
]
salary_data_df = pd.DataFrame(salary_data_raw)

# Time to Correct Data (Simplified)
time_to_correct_df = pd.DataFrame([{"Average": 40}]) # 40 hours average

print("✅ Resource data (Role Groups, Salary, Time) loaded.")

## 4. Create Mappings

In [ ]:
def create_role_mappings(role_groups_df):
    """Create mappings for roles to groups."""
    role_to_group = {}
    for _, row in role_groups_df.iterrows():
        group_name = row['Group Name']
        roles = [r.strip() for r in row['Roles in Group'].split(',')]
        for role in roles:
            role_to_group[role.lower()] = group_name
    return role_to_group

role_to_group_map = create_role_mappings(role_groups_df)
print(f"✅ Created role-to-group mapping for {len(role_to_group_map)} roles.")

## 5. Load Input Data (Sample JDs and Classifications)

In [ ]:
# Sample JDs
sample_jds_data = pd.read_csv(io.StringIO("""
Job Code,Job Title,Position Summary,Education,Work Experience,Essential Functions,Licenses and Certifications,"Knowledge, Skills and Abilities"
82222,Spec Migrant Youth,"The Migrant Youth Specialist is responsible for coordinating, creating, and implementing supports for migrant youth and their families. This position requires coordinating resources and partnerships in conjunction with diverse community stakeholders and providing direct service to migrant youth and their families. The Migrant Youth Specialist serves as a liaison to contracted vendors, and other departments to ensure compliance with state and federal laws, regulations and guidelines for the use of federal funds and administration of state and federal grants.",Bachelor's Degree  Preferred, Experience working with migrant youth and families and Migrant Education Programs (MEP) Preferred,"Conduct individual needs assessments (INAs) for migrant youth monthly and as needed; document and submit related reports
Coordinate services and supports for migrant youth as identified through the individual needs assessments
Partner with other support services, such as: social workers, counselors,
translators, and other personnel to support migrant youth
Coordinate and maintain data for migrant students monthly and as needed; work
with MNPS Migrant Coordinator to maintain and submit data to Tennessee Migrant
Education Program (TN MEP) monthly and as needed
Travel within district to facilitate direct support and collect data at the school level, as necessary
Assist in the identification, recruitment, and provision of services for migrant youth in the area, as needed
Monitor data quality through data quality monitoring and provide feedback as needed
Maintain confidential files according to protocols and requirements
Perform other related duties as assigned",None,"Interpersonal Skills--maintain confidentiality, work well with others, and establish rapport with students in order to strengthen the relationship between student and service provider; problem solve and help facilitate solutions; address problems in a courteous, solution-focused manner when working with students and families
Communication Skills--maintain positive communication with students,
families, and colleagues; ability to communicate in Spanish (read, write, speak, and translate) preferred but not required
Organization--work at multiple sites and with various organizational cultures; maintain flexibility in work schedule; maintain accurate records on students and program activities; maintain flexible work schedule to meet the needs of students and families
Personal Characteristics--exhibit professionalism and maintain high ethical
standards; maintain confidentiality; serve as a positive role model; take on
additional tasks as needed; complete assigned duties and have reliable attendance"
28460,Coor Athletic Operations,"The Athletic Operations Coordinator is responsible for overseeing operations related to Metro Nashville Public Schools (MNPS) intercollegiate athletics programs. This position assists the Executive Director of Athletics with business management tasks to align operations with the goals of MNPS athletics.","Bachelor's degree in sports management, business administration, education, kinesiology or related field Preferred", 3 years Minimum Experience in athletic program operations or coaching at the collegiate level preferred OR Equivalent combination of related training and experience Minimum,"Assist in managing athletics programs' schedules and serve as liaison for officials, coaches, and district athletics operations employees
Review and approve or deny visiting teams' travel plans including hotel accommodations, scouting, etc.
Prepare and manage the athletics budget, track all budgetary expenses, and generate monthly reports for fiscal audits
Complete research to develop program processes that increase productivity and fiscal responsibility
Evaluate program processes and work with athletics leadership to determine which processes to strengthen or discontinue
Develop new strategies for securing external funding; maintain and nurture existing donor relationships
Maintain facilities schedule and assign specific fields or gyms for team practices, games, and events
Maintain inventory of MNPS athletics equipment
Assist with hiring, evaluating, and supervising coaches; conduct exit interviews
Communicate directly with district leadership on legal, student welfare, and public relations issues pertaining to athletics
Evaluate the effects of the athletic program on the overall district mission; draft program-related reports to assess that alignment
Participate in professional development activities pertaining to district athletics and sports safety
Perform other related duties as assigned",None,"Interpersonal Skills--maintain confidentiality; cultivate positive working
relationships with coaches, officials, parents, alumni, and staff members
Communication Skills--communicate effectively in verbal and written forms; actively listen to conversations and explain solutions to problems in an understandable manner; provide high-quality customer service to stakeholders
Organization--effectively manage time and maintain accurate and organized records; delegate tasks when necessary; manage multiple assignments and meet project deadlines
Personal Characteristics--maintain confidentiality; assume responsibility for
errors and develop a growth plan to rectify performance; work independently and
as a member of a team; be attentive and detail-oriented; maintain reliable
attendance; demonstrate flexibility"
"""))

# Sample Classifications
sample_classifications_data = pd.read_csv(io.StringIO("""
Job Code,Classification,Sub Group,Justification
82222,Specialist,None,"The job involves specialized knowledge in supporting migrant youth and families, coordinating multiple services, acting as a liaison with community stakeholders, and complying with federal grant regulations. These tasks align with the Specialist role, which requires advanced, specialized knowledge to provide expert support, often in coordination with community partners and multiple organizations. The focus on direct service, coordination with diverse partners, and technical knowledge regarding migrant education programs fits the Specialist category."
28460,Coordinator,None,"The role focuses on coordinating athletic programs, managing schedules, and overseeing operations, which aligns with the Coordinator classification. The individual in this position works across multiple areas of athletic operations, serving as a liaison between various parties including coaches, officials, and district leadership. These tasks are typical of a Coordinator, who organizes, manages, and ensures the smooth operation of specific programs. The responsibilities do not rise to the level of Manager or Director, as they involve operational coordination rather than strategic management."
"""))

print(f"✅ Loaded {len(sample_jds_data)} job descriptions")
print(f"✅ Loaded {len(sample_classifications_data)} classifications")

## 6. Merge Data

In [ ]:
# Merge JDs with classifications
merged_df = sample_jds_data.merge(sample_classifications_data, on='Job Code', how='inner')
print(f"✅ Merged {len(merged_df)} records")

## 7. Core Scoring Functions

In [ ]:
def preprocess_text(text):
    """Preprocess text for similarity calculation."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

def calculate_ksac_similarity(jd_text, role_name):
    """
    Calculate KSAC similarity score using TF-IDF and cosine similarity.
    Returns a score from 0 to 1.
    """
    if not jd_text or not role_name:
        return 0.5  # Neutral score if missing data
    
    try:
        vectorizer = TfidfVectorizer(stop_words='english', max_features=100)
        tfidf_matrix = vectorizer.fit_transform([jd_text, role_name])
        similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return max(0.0, min(1.0, similarity))  # Ensure between 0 and 1
    except:
        return 0.5  # Return neutral score on error

def calculate_salary_impact(classification, salary_data_df, role_to_group_map):
    """
    Calculate salary impact score based on role group mapping.
    Returns a normalized score from 0 to 1.
    """
    if pd.isna(classification):
        return 0.5
    
    role_group = role_to_group_map.get(classification.lower())
    if not role_group:
        return 0.5  # Neutral score if role group not found
    
    salary_row = salary_data_df[salary_data_df['Major Role Grouping'] == role_group]
    if salary_row.empty:
        return 0.5
    
    avg_salary = salary_row.iloc[0]['Average Annual Salary']
    min_salary = salary_data_df['Average Annual Salary'].min()
    max_salary = salary_data_df['Average Annual Salary'].max()
    
    normalized_score = (avg_salary - min_salary) / (max_salary - min_salary)
    return max(0.0, min(1.0, normalized_score))

def calculate_time_impact(time_to_correct_df):
    """
    Calculate time impact score.
    Returns a normalized score from 0 to 1 based on average correction time.
    """
    avg_time = time_to_correct_df.iloc[0]['Average']
    # Normalize: assume 0 hours = 1.0 (no correction needed), 80 hours = 0.0 (significant correction)
    max_time = 80
    normalized_score = 1.0 - (avg_time / max_time)
    return max(0.0, min(1.0, normalized_score))

def calculate_subgroup_score(sub_group):
    """
    Calculate sub-group score.
    Returns 1.0 if sub-group is present and meaningful, 0.5 if None or empty.
    """
    if pd.isna(sub_group) or str(sub_group).strip().lower() in ['none', '']:
        return 0.5
    return 1.0

def calculate_justification_score(justification):
    """
    Calculate justification quality score based on length and content.
    Returns a score from 0 to 1.
    """
    if pd.isna(justification):
        return 0.3
    
    just_text = str(justification).strip()
    if not just_text:
        return 0.3
    
    # Score based on length (longer justifications tend to be more detailed)
    word_count = len(just_text.split())
    length_score = min(word_count / 100, 1.0)  # Cap at 100 words = 1.0
    
    # Score based on presence of key phrases
    key_phrases = ['align', 'focus', 'require', 'involve', 'coordinate', 'manage', 'specialist', 'expert']
    phrase_score = sum(phrase in just_text.lower() for phrase in key_phrases) / len(key_phrases)
    
    # Combine scores (70% length, 30% content)
    final_score = (0.7 * length_score) + (0.3 * phrase_score)
    return max(0.3, min(1.0, final_score))  # Minimum 0.3 if justification exists

def calculate_confidence_score(component_scores):
    """
    Calculate confidence score based on consistency of component scores.
    Returns a score from 0 to 1.
    """
    if not component_scores:
        return 0.5
    
    # Calculate standard deviation of component scores
    std_dev = np.std(component_scores)
    # Low std dev = high confidence, high std dev = low confidence
    # Assuming std dev ranges from 0 (all scores identical) to ~0.5 (very diverse)
    confidence = 1.0 - min(std_dev / 0.5, 1.0)
    return max(0.0, min(1.0, confidence))

def detect_borderline_case(confidence_score, likelihood_score, classification, jd_text, role_to_group_map):
    """
    Detect if a classification is borderline and suggest alternatives.
    Returns (is_borderline, alternative_role, suggestion)
    """
    is_borderline = confidence_score < CONFIDENCE_THRESHOLD
    alternative = None
    suggestion = None
    
    if is_borderline and ENABLE_JD_SUGGESTIONS:
        # Analyze JD to suggest improvements
        current_group = role_to_group_map.get(classification.lower())
        
        # Find alternative roles with high similarity
        similarities = {}
        for role, group in role_to_group_map.items():
            if group != current_group:  # Only consider different groups
                sim = calculate_ksac_similarity(jd_text, role)
                similarities[role] = sim
        
        if similarities:
            best_alt = max(similarities.items(), key=lambda x: x[1])
            if best_alt[1] > 0.6:  # Only suggest if similarity is reasonably high
                alternative = best_alt[0].title()
                suggestion = f"Consider reviewing classification. JD shows similarity to '{alternative}' role. "
                suggestion += "Clarify key responsibilities and required expertise level in JD."
    
    return is_borderline, alternative, suggestion

def calculate_likelihood_score(ksac_sim, salary_imp, time_imp, subgroup, justification):
    """
    Calculate the weighted likelihood score on a 1-5 scale.
    """
    weighted_sum = (
        ksac_sim * WEIGHT_KSAC_SIMILARITY +
        salary_imp * WEIGHT_SALARY_IMPACT +
        time_imp * WEIGHT_TIME_IMPACT +
        subgroup * WEIGHT_SUBGROUP +
        justification * WEIGHT_JUSTIFICATION
    )
    
    # Convert from 0-1 scale to 1-5 scale
    likelihood = 1 + (weighted_sum * 4)
    return max(1.0, min(5.0, likelihood))

def convert_to_accuracy(likelihood_score):
    """
    Convert likelihood score (1-5) to accuracy percentage (0-100%).
    Mapping: 1.0 = 0%, 5.0 = 100%
    """
    accuracy = ((likelihood_score - 1) / 4) * 100
    return max(0.0, min(100.0, accuracy))

def categorize_performance(likelihood_score):
    """
    Categorize performance based on likelihood score compared to human baseline.
    """
    accuracy = convert_to_accuracy(likelihood_score)
    
    if accuracy >= 97:  # ~4.88 likelihood
        return "Superhuman"
    elif accuracy >= HUMAN_BASELINE_MAX:  # ~4.76 likelihood
        return "Above Human"
    elif accuracy >= HUMAN_BASELINE_MIN:  # ~4.52 likelihood
        return "Human-level"
    elif accuracy >= 70:  # ~3.8 likelihood
        return "Below Human"
    elif accuracy >= 50:  # ~3.0 likelihood
        return "Poor"
    else:
        return "Critical"

def determine_review_priority(likelihood_score, confidence_score, is_borderline, has_suggestion):
    """
    Determine review priority based on likelihood, confidence, and borderline status.
    Aligned with report generation logic.
    """
    # Below Human (<4.0 likelihood, <88% accuracy)
    if likelihood_score < 4.0:
        # If borderline with suggestion, it's High priority
        if is_borderline and has_suggestion:
            return REVIEW_PRIORITY_HIGH
        # If low confidence with suggestion, also High priority
        elif confidence_score < CONFIDENCE_THRESHOLD and has_suggestion:
            return REVIEW_PRIORITY_HIGH
        # Otherwise, it's Critical (needs immediate review)
        else:
            return REVIEW_PRIORITY_CRITICAL
    
    # Human-level (4.0 <= likelihood < 4.5, 88-93% accuracy)
    elif likelihood_score < 4.5:
        # If low confidence, needs Medium priority review
        if confidence_score < CONFIDENCE_THRESHOLD:
            return REVIEW_PRIORITY_MEDIUM
        # Otherwise, Low priority (acceptable performance)
        else:
            return REVIEW_PRIORITY_LOW
    
    # Above Human (>=4.5 likelihood, >=93% accuracy)
    else:
        return REVIEW_PRIORITY_LOW

print("✅ Core scoring functions defined.")

## 8. Analyze Each Classification

In [ ]:
results = []

for _, row in merged_df.iterrows():
    # Prepare JD text
    jd_text = preprocess_text(
        str(row.get('Position Summary', '')) + ' ' +
        str(row.get('Essential Functions', '')) + ' ' +
        str(row.get('Knowledge, Skills and Abilities', ''))
    )
    
    classification = row.get('Classification', '')
    sub_group = row.get('Sub Group', '')
    justification = row.get('Justification', '')
    
    # Calculate component scores
    ksac_sim = calculate_ksac_similarity(jd_text, classification)
    salary_imp = calculate_salary_impact(classification, salary_data_df, role_to_group_map)
    time_imp = calculate_time_impact(time_to_correct_df)
    subgroup_score = calculate_subgroup_score(sub_group)
    just_score = calculate_justification_score(justification)
    
    # Calculate likelihood and confidence
    likelihood = calculate_likelihood_score(ksac_sim, salary_imp, time_imp, subgroup_score, just_score)
    component_scores = [ksac_sim, salary_imp, time_imp, subgroup_score, just_score]
    confidence = calculate_confidence_score(component_scores)
    
    # Detect borderline cases
    is_borderline, alt_role, suggestion = detect_borderline_case(
        confidence, likelihood, classification, jd_text, role_to_group_map
    )
    
    # Determine review priority
    review_priority = determine_review_priority(
        likelihood, confidence, is_borderline, suggestion is not None
    )
    
    # Store results
    result = {
        'job_code': row.get('Job Code'),
        'job_title': row.get('Job Title'),
        'classification': classification,
        'sub_group': sub_group,
        'ksac_similarity': ksac_sim,
        'salary_impact': salary_imp,
        'time_impact': time_imp,
        'subgroup_score': subgroup_score,
        'justification_score': just_score,
        'likelihood_score': likelihood,
        'accuracy_equivalent': convert_to_accuracy(likelihood),
        'confidence_score': confidence,
        'is_borderline': is_borderline,
        'borderline_alternative': alt_role,
        'jd_suggestion': suggestion,
        'review_priority': review_priority
    }
    results.append(result)

results_df = pd.DataFrame(results)
results_df['performance_category'] = results_df['likelihood_score'].apply(categorize_performance)

print(f"✅ Analyzed {len(results_df)} records")

## 9. Generate Analysis Report

In [ ]:
def generate_analysis_report(results_df):
    """Generate a detailed analysis report."""
    report_lines = []
    report_lines.append("=" * 80)
    report_lines.append("MNPS ENHANCED LIKELIHOOD SCORER - ANALYSIS REPORT")
    report_lines.append(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append("=" * 80)
    report_lines.append("")
    report_lines.append("CONFIGURATION:")
    report_lines.append(f"  Human Baseline Range: {HUMAN_BASELINE_MIN}%-{HUMAN_BASELINE_MAX}%")
    report_lines.append(f"  Confidence Threshold: {CONFIDENCE_THRESHOLD}")
    report_lines.append(f"  Borderline Detection: {'Enabled' if ENABLE_BORDERLINE_DETECTION else 'Disabled'}")
    report_lines.append(f"  JD Suggestions: {'Enabled' if ENABLE_JD_SUGGESTIONS else 'Disabled'}")
    report_lines.append("")
    report_lines.append("OVERALL STATISTICS:")
    report_lines.append(f"  Total Records Analyzed: {len(results_df)}")
    report_lines.append(f"  Average Likelihood Score: {results_df['likelihood_score'].mean():.2f}")
    report_lines.append(f"  Average Accuracy Equivalent: {results_df['accuracy_equivalent'].mean():.1f}%")
    report_lines.append(f"  Average Confidence Score: {results_df['confidence_score'].mean():.2f}")
    report_lines.append("")

    # Performance Breakdown (Fixed: Use likelihood score, not performance category for counts)
    report_lines.append("PERFORMANCE BREAKDOWN (by Likelihood Score):")
    report_lines.append(f"  Superhuman (≥4.75): {(results_df['likelihood_score'] >= 4.75).sum()} ({(results_df['likelihood_score'] >= 4.75).sum()/len(results_df)*100:.1f}%)")
    report_lines.append(f"  Excellent (4.5-4.75): {((results_df['likelihood_score'] >= 4.5) & (results_df['likelihood_score'] < 4.75)).sum()} ({((results_df['likelihood_score'] >= 4.5) & (results_df['likelihood_score'] < 4.75)).sum()/len(results_df)*100:.1f}%)")
    report_lines.append(f"  Human-level (4.0-4.5): {((results_df['likelihood_score'] >= 4.0) & (results_df['likelihood_score'] < 4.5)).sum()} ({((results_df['likelihood_score'] >= 4.0) & (results_df['likelihood_score'] < 4.5)).sum()/len(results_df)*100:.1f}%)")
    report_lines.append(f"  Below Human (<4.0): {(results_df['likelihood_score'] < 4.0).sum()} ({(results_df['likelihood_score'] < 4.0).sum()/len(results_df)*100:.1f}%)")
    report_lines.append("")

    report_lines.append("CONFIDENCE DISTRIBUTION:")
    high_conf = (results_df['confidence_score'] >= 0.8).sum()
    mod_conf = ((results_df['confidence_score'] >= CONFIDENCE_THRESHOLD) & (results_df['confidence_score'] < 0.8)).sum()
    low_conf = (results_df['confidence_score'] < CONFIDENCE_THRESHOLD).sum()
    report_lines.append(f"  High (≥0.8): {high_conf} ({high_conf/len(results_df)*100:.1f}%)")
    report_lines.append(f"  Moderate ({CONFIDENCE_THRESHOLD}-0.8): {mod_conf} ({mod_conf/len(results_df)*100:.1f}%)")
    report_lines.append(f"  Low (<{CONFIDENCE_THRESHOLD}): {low_conf} ({low_conf/len(results_df)*100:.1f}%) - FLAGGED")
    report_lines.append("")

    # Review Priority Breakdown (Fixed: Use the correct priority calculation)
    report_lines.append("REVIEW PRIORITY BREAKDOWN:")
    for priority in [REVIEW_PRIORITY_CRITICAL, REVIEW_PRIORITY_HIGH, REVIEW_PRIORITY_MEDIUM, REVIEW_PRIORITY_LOW]:
        count = (results_df['review_priority'] == priority).sum()
        pct = count / len(results_df) * 100
        report_lines.append(f"  {priority}: {count} ({pct:.1f}%)")
    report_lines.append("")

    # Borderline Cases Summary
    borderline_count = results_df['is_borderline'].sum()
    with_alternatives = results_df['borderline_alternative'].notna().sum()
    report_lines.append("BORDERLINE CASES SUMMARY:")
    report_lines.append(f"  Total Borderline Cases: {borderline_count}")
    report_lines.append(f"  With Alternative Suggestions: {with_alternatives}")
    report_lines.append("")

    # Top and Bottom Performers
    report_lines.append("TOP 5 PERFORMERS (by likelihood score):")
    top_5 = results_df.nlargest(5, 'likelihood_score')
    for _, row in top_5.iterrows():
        report_lines.append(f"  {row['likelihood_score']:.2f} - {row['job_title'][:50]} → {row['classification']}")
    report_lines.append("")

    report_lines.append("BOTTOM 5 PERFORMERS (by likelihood score):")
    bottom_5 = results_df.nsmallest(5, 'likelihood_score')
    for _, row in bottom_5.iterrows():
        report_lines.append(f"  {row['likelihood_score']:.2f} - {row['job_title'][:50]} → {row['classification']}")
    report_lines.append("")

    # Recommendations
    report_lines.append("=" * 80)
    report_lines.append("RECOMMENDATIONS:")
    report_lines.append("=" * 80)
    report_lines.append("")
    report_lines.append("1. REVIEW PRIORITY:")
    report_lines.append(f"   - Start with {results_df[results_df['review_priority'] == REVIEW_PRIORITY_CRITICAL].shape[0]} Critical priority cases (likelihood <4.0 and high confidence OR no suggestion needed)")
    report_lines.append(f"   - Then review {results_df[results_df['review_priority'] == REVIEW_PRIORITY_HIGH].shape[0]} High priority cases (likelihood <4.0 and borderline/low confidence with suggestions)")
    report_lines.append(f"   - Quick check on {results_df[results_df['review_priority'] == REVIEW_PRIORITY_MEDIUM].shape[0]} Medium priority cases")
    report_lines.append(f"   - Fast-track approve {results_df[results_df['review_priority'] == REVIEW_PRIORITY_LOW].shape[0]} Low priority cases")
    report_lines.append("")
    report_lines.append("2. BORDERLINE CASES:")
    if borderline_count > 0:
        report_lines.append(f"   - Review detailed borderline report for {borderline_count} cases")
        report_lines.append(f"   - {with_alternatives} cases have alternative role suggestions")
        report_lines.append("   - Share JD improvement suggestions with hiring departments")
    else:
        report_lines.append("   - No borderline cases detected - excellent!")
    report_lines.append("")
    report_lines.append("3. EXPECTED TIME SAVINGS:")
    fast_track = results_df[results_df['review_priority'] == REVIEW_PRIORITY_LOW].shape[0]
    report_lines.append(f"   - {fast_track} cases ({fast_track/len(results_df)*100:.1f}%) can be fast-tracked")
    report_lines.append(f"   - Estimated review time reduction: 25-30%")
    report_lines.append("")

    return "\n".join(report_lines)

report_text = generate_analysis_report(results_df)
print("✅ Analysis report generated.")

## 10. Output Results

In [ ]:
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
print(f"Total Records: {len(results_df)}")
print(f"Average Likelihood: {results_df['likelihood_score'].mean():.2f}")
print(f"Average Accuracy: {results_df['accuracy_equivalent'].mean():.1f}%")
print(f"Average Confidence: {results_df['confidence_score'].mean():.2f}")
print(f"Human Baseline: {HUMAN_BASELINE_MIN}-{HUMAN_BASELINE_MAX}%")
print(f"Records Below Human (<4.0): {(results_df['likelihood_score'] < 4.0).sum()}")
print(f"Records Above Human (≥4.0): {(results_df['likelihood_score'] >= 4.0).sum()}")
print(f"Borderline Cases (<{CONFIDENCE_THRESHOLD}): {(results_df['confidence_score'] < CONFIDENCE_THRESHOLD).sum()}")
print(f"Critical Priority: {(results_df['review_priority'] == REVIEW_PRIORITY_CRITICAL).sum()}")
print(f"High Priority: {(results_df['review_priority'] == REVIEW_PRIORITY_HIGH).sum()}")
print(f"Medium Priority: {(results_df['review_priority'] == REVIEW_PRIORITY_MEDIUM).sum()}")
print(f"Low Priority: {(results_df['review_priority'] == REVIEW_PRIORITY_LOW).sum()}")

# Display sample of results
print("\n📋 Sample of Enhanced Results:")
print(results_df[['job_title', 'classification', 'sub_group', 'likelihood_score', 'confidence_score', 'review_priority', 'performance_category']].head())

## 11. Display Full Analysis Report

In [ ]:
print("\n" + "="*80)
print("FULL ANALYSIS REPORT")
print("="*80)
print(report_text)

## 12. Optional: Save Results to CSV

In [ ]:
# Uncomment to save results
# results_df.to_csv('enhanced_likelihood_scores.csv', index=False)
# with open('analysis_report.txt', 'w') as f:
#     f.write(report_text)

print("\n✅ Notebook execution complete.")